# Qwen3.5-0.8B — ARDB unified page understanding (bbox + text, one forward pass) (Unsloth, LoRA, T4/L4)

Fine-tunes `unsloth/Qwen3.5-0.8B` on `Soxavin/ardb-sft-v5` — same one-task, one-forward-pass
design as the Gemma 4 E2B notebook this mirrors (`colab_gemma4_e2b_finetune.ipynb`): full page image
in, one JSON list out, where every region carries its `box_2d`, `label`, AND its transcribed `text`
together:

```json
[
  {"box_2d": [15,15,90,90], "label": "Picture", "text": ""},
  {"box_2d": [21,91,51,266], "label": "Page-Furniture", "text": "ធនាគារ ARDB"},
  {"box_2d": [111,15,885,984], "label": "Table", "text": "| ២៣ | ... |"},
  {"box_2d": [889,17,983,385], "label": "Section-Header", "text": "ខែមករា ថ្ងៃទី26"}
]
```

**Reusing the model-agnostic dataset repo, not a Gemma-only one**: this is the exact same
`box_2d` (`[y1, x1, y2, x2]` normalized 0-1000), same instruction wording, same schema
(`image`/`instruction`/`text`) as the Gemma notebook consumes. Qwen3.5-0.8B's own public docs
don't specify a confirmed grounding-output convention (search results only cover the older,
separate Qwen3-VL family) — but since this is a *fine-tune*, not zero-shot prompting, we don't
need to match whatever convention the base model already knows; training teaches it our format
directly. No coordinate conversion or dataset rebuild needed.

`Soxavin/ardb-sft-v5` (47 non-frozen documents / 128 pages: 101 train / 9 validation / 18 test)
stratifies the split by structural template era on top of date-clustering, so both bulletin
layouts are represented in every split. Training images get the same light train-only
augmentation as the Gemma notebook (brightness/contrast jitter + occasional blur) —
validation/test images are never touched by it.

Model repo, LoRA rank, batch size, and omitting `get_chat_template()` are all taken directly
from Unsloth's own official `Qwen3_5_(0_8B)_Vision.ipynb` notebook (fetched 2026-08-03), not
guessed by analogy to Qwen3-VL or Gemma. QLoRA (4-bit) is deliberately NOT used here — Unsloth's
own Qwen3.5 fine-tuning guide explicitly recommends against 4-bit training for this model family
due to quantization differences; this notebook already does 16-bit LoRA (`load_in_4bit=False`),
matching that guidance.

**Known base-model limitation (read before running a full training run)**: an earlier trial on
this project confirmed the *base* model (zero fine-tuning steps) generates fluent Thai script
instead of Khmer, on both a plain translation prompt and this notebook's real image+instruction
task — see `docs/PROJECT_LOG.md` §2.104. This run is being repeated on the larger, era-stratified
`ardb-sft-v5` dataset anyway — both because the mentor specifically wants this model fine-tuned
(a result worth having and presenting regardless of outcome) and to dig further into the root
cause. The eval cell below now reports script (Khmer/Thai/neither) **per JSON region**, not just
per validation row, cross-checked with `fast-langdetect` alongside the existing Unicode-codepoint
check — not because more training data is expected to fix a pretraining-time gap on its own, but
so the failure (if it persists) is precisely measured and presentable, not just asserted.

**A note on training speed**: Qwen3.5 uses custom Mamba/Triton kernels (its Gated DeltaNet
hybrid-attention backbone) that can take noticeably longer to compile than a plain transformer,
especially on a T4 — if the first training step looks stuck for several minutes with no output,
that's very likely kernel compilation, not a hang.

**Steps:** Runtime ▸ Change runtime type ▸ **T4 or L4 GPU** → run all cells with
`SMOKE_TEST = True` first (10 examples, ~a few minutes, plus first-run kernel compile time) to
confirm nothing crashes/OOMs → then set `SMOKE_TEST = False` and run again for the full training
run. To sweep epoch count, change `EPOCHS` in the cell below and rerun from there — each value
pushes its adapter to its own repo so runs don't overwrite each other.

In [ ]:
%%capture
import os, importlib.util
# History of this cell's torch/transformers pins:
# 1) Official Qwen3.5 (0.8B) Vision notebook pins torch==2.8.0, transformers==5.2.0 ->
#    threw ImportError: cannot import name 'ScalingType' from 'torch.nn.functional'
#    (confirmed via direct PyTorch source inspection: ScalingType exists in torch==2.10.0,
#    not 2.8.0/2.9.0).
# 2) Bumped torch to 2.10.0 -> fixed that, but unmasked ImportError: cannot import name
#    'is_opentelemetry_available' from transformers.utils.import_utils (confirmed present in
#    transformers==5.2.0's own source, so not a missing-symbol problem).
# 3) Stopped force-pinning torch at all -- floats to whatever Colab provides, matching this
#    project's Gemma notebook's proven-working pattern (Gemma never pins/reinstalls torch
#    either) -> fixed that error too, but unmasked a THIRD: ImportError: cannot import name
#    'CUSTOM_KEY' from torch.ao.quantization.
# 4) This version: three different symbols failing in the exact same except-block, each time
#    torch changes, is a systemic-drift signal, not three separate bugs -- transformers==
#    5.2.0 (from the official Qwen3.5 notebook) simply predates whatever torch API surface
#    a floating/current torch now presents. The fix is to stop pinning the OTHER stale half
#    of the pair too: bumped transformers 5.2.0 -> 5.5.0, matching Gemma's own already-proven
#    transformers pin exactly (paired with the same floating-torch approach). Confirmed via
#    direct source check that transformers==5.5.0 still ships the Qwen3.5 model module
#    (models/qwen3_5/) before making this change.
!pip install --upgrade -qqq uv
if "COLAB_" in "".join(os.environ.keys()):
    import re
    import torch
    v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10': '0.0.34', '2.9': '0.0.33.post1', '2.8': '0.0.32.post2'}.get(v, "0.0.34")
    !uv pip install -qqq \
        "triton>=3.3.0" bitsandbytes {xformers} \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("torch") is None:
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        torch "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.5.0
# causal_conv1d's prebuilt wheel matches torch==2.8.0 specifically. Since torch is no longer
# pinned here, it will very likely build from source instead -- slower (~10 min), not
# broken; the official notebook's own comment on this line already anticipates this for
# "newer torch versions".
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0
import torch
if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
    !uv pip install --no-deps "apache-tvm-ffi==0.1.9" "tilelang==0.1.8"
else:
    os.environ["FLA_TILELANG"] = "0"
!uv pip install --no-deps --upgrade "torchao>=0.16.0"
# fast-langdetect: our own addition, not in the official notebook -- only used by the eval
# cell's script-detection cross-check, not training itself. A genuinely separate, unrelated
# package (fasttext-based language ID) from everything else in this cell.
!pip install -qqq fast-langdetect


In [ ]:
# Outside %%capture on purpose, unlike the install cell above -- prints what actually got
# installed instead of assuming it. unsloth/unsloth_zoo now install from git main (see the
# install cell's comment for why), so there's no fixed expected version to assert against
# here anymore -- if this cell itself throws (rather than just printing something), the
# ScalingType error is back and worth reporting with these exact version numbers attached.
import torch, unsloth, unsloth_zoo, transformers
print(f"torch={torch.__version__} unsloth={unsloth.__version__} "
     f"unsloth_zoo={unsloth_zoo.__version__} transformers={transformers.__version__}")

In [ ]:
from unsloth import FastVisionModel
import torch

# unsloth/Qwen3.5-0.8B, load_in_4bit=False, no device_map override: all three confirmed
# against Unsloth's own official Qwen3.5 (0.8B) Vision notebook, not assumed from Gemma's
# setup. Gemma needed device_map={"": 0} to work around a bitsandbytes 4-bit dispatch bug
# (load_in_4bit=True there) -- that bug is specific to 4-bit loading, so it doesn't apply
# here since Qwen3.5 loads in 16-bit.
model, processor = FastVisionModel.from_pretrained(
    "unsloth/Qwen3.5-0.8B",
    load_in_4bit = False,               # 16-bit LoRA, not QLoRA -- Unsloth's own default for this model
    use_gradient_checkpointing = "unsloth",
)


In [ ]:
# Base-model script check (BEFORE any LoRA/training): re-verifies the §2.104 finding
# (base model generates Thai instead of Khmer) against THIS load, so if it's ever fixed
# upstream (a newer unsloth/Qwen3.5 release) or was somehow environment-specific, that
# shows up here directly instead of being assumed still true. This is Probe 1 of 2 -- a
# plain-text translation prompt, no image involved, which isolates whether the gap is in
# the language model at all, or specific to the vision path. Probe 2 (the real
# image+instruction task) runs later, right after the dataset loads.
import re as _re
_KHMER_RE_PROBE = _re.compile(r"[ក-៿]")
_THAI_RE_PROBE = _re.compile(r"[฀-๿]")

def _script_of(text: str) -> str:
    k, t = len(_KHMER_RE_PROBE.findall(text)), len(_THAI_RE_PROBE.findall(text))
    if k == 0 and t == 0:
        return "neither (no Khmer or Thai codepoints)"
    return "khmer" if k >= t else f"thai ({t} Thai vs {k} Khmer codepoints)"

FastVisionModel.for_inference(model)

_probe_messages = [{"role": "user", "content": [
    {"type": "text", "text": "Translate this to Khmer: The price of rice today is 4500 riels per kilogram."}]}]
_probe_input = processor.apply_chat_template(_probe_messages, add_generation_prompt=True)
_probe_inputs = processor(None, _probe_input, add_special_tokens=False, return_tensors="pt").to("cuda")
_probe_out = model.generate(**_probe_inputs, max_new_tokens=200, use_cache=True, do_sample=False)
_probe_text = processor.decode(_probe_out[0][_probe_inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("--- Probe 1: plain-text Khmer translation (no image, no LoRA yet) ---")
print(_probe_text)
print(f"script: {_script_of(_probe_text)}\n")

FastVisionModel.for_training(model)  # restore training mode for the cells below


In [ ]:
# LoRA adapters on top of the frozen 16-bit base. r/alpha = 16 (not Gemma's 32) matches
# Unsloth's own default for this model -- 0.8B has far less capacity to begin with, so a
# larger rank is more likely to overfit our already-small dataset than help it.
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)


## Data

One flat dataset, one schema — `image` + `instruction` + `text` (the JSON list of regions). Same repo the Gemma notebook trains on (see the top-of-notebook note on why no format conversion is needed).

In [ ]:
SMOKE_TEST = True  # flip to False only after a smoke run has completed without errors

# Epoch sweep knob (see SFTConfig cell below): not yet re-verified for this model/size --
# the Gemma run log found more epochs made JSON-parse-failure rate WORSE on a small dataset
# (a classic overfitting signature), but Qwen3.5-0.8B has a different capacity/LoRA rank, so
# treat 3 as a starting point to sweep (try 2, 3, 5, ...), not a settled number. Change this
# and rerun from here; each value pushes its adapter to its own repo (see the push-to-hub
# cell) so sweep runs don't overwrite each other. Log each run's result in
# eval/qwen_finetune_runs.csv before changing this again.
EPOCHS = 3

from datasets import load_dataset

_REPO_ID = "Soxavin/ardb-sft-v5"

dataset = load_dataset(_REPO_ID, split="train")
val_dataset = load_dataset(_REPO_ID, split="validation")

if SMOKE_TEST:
    dataset = dataset.select(range(min(10, len(dataset))))
    val_dataset = val_dataset.select(range(min(5, len(val_dataset))))

print(f"train rows: {len(dataset)}, validation rows: {len(val_dataset)}")


In [ ]:
import random
from PIL import Image, ImageEnhance, ImageFilter

# Train-only augmentation, identical to the Gemma notebook's: brightness/contrast jitter +
# light blur, applied only to converted_dataset below (never val_dataset -- the eval cells
# read val_dataset's images directly, never through convert_to_conversation). Deliberately
# no rotation/affine/scale/crop: box_2d targets are ground-truth coordinates the model must
# reproduce, and any geometric transform here would need every box_2d in the target JSON
# re-projected to match, which this pass doesn't do.
_aug_rng = random.Random(3407)

def augment_image(img: Image.Image, rng: random.Random) -> Image.Image:
    img = ImageEnhance.Brightness(img).enhance(1.0 + rng.uniform(-0.1, 0.1))
    img = ImageEnhance.Contrast(img).enhance(1.0 + rng.uniform(-0.1, 0.1))
    if rng.random() < 0.2:
        img = img.filter(ImageFilter.GaussianBlur(radius=rng.uniform(0.1, 0.3)))
    return img

In [ ]:
# Base-model script check, Probe 2 of 2: the real image+instruction task (this notebook's
# actual training format). LoRA adapters were already attached by the cell above, but
# freshly-initialized LoRA adapters are a mathematical no-op until trained (the standard
# `lora_alpha` scaling means an untrained adapter contributes ~0 to the forward pass) --
# so this still measures base-model behavior, before trainer.train() has updated any
# weights. Isolates whether an image input changes the script-choice finding from Probe 1
# (plain text).
FastVisionModel.for_inference(model)

_probe2_sample = val_dataset[0]
_probe2_messages = [{"role": "user", "content": [
    {"type": "image"}, {"type": "text", "text": _probe2_sample["instruction"]}]}]
_probe2_input = processor.apply_chat_template(_probe2_messages, add_generation_prompt=True)
_probe2_inputs = processor(_probe2_sample["image"], _probe2_input, add_special_tokens=False,
                           return_tensors="pt").to("cuda")
_probe2_out = model.generate(**_probe2_inputs, max_new_tokens=500, use_cache=True, do_sample=False)
_probe2_text = processor.decode(_probe2_out[0][_probe2_inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("--- Probe 2: real image+instruction task (base model, no LoRA training yet) ---")
print(_probe2_text[:1000])
print(f"script: {_script_of(_probe2_text)}\n")

FastVisionModel.for_training(model)  # restore training mode for the cells below


In [ ]:
def convert_to_conversation(sample, train: bool = False, rng: random.Random | None = None):
    img = augment_image(sample["image"], rng) if train else sample["image"]
    return {"messages": [
        {"role": "user", "content": [
            {"type": "text", "text": sample["instruction"]},
            {"type": "image", "image": img},
        ]},
        {"role": "assistant", "content": [{"type": "text", "text": sample["text"]}]},
    ]}

converted_dataset = [convert_to_conversation(s, train=True, rng=_aug_rng) for s in dataset]


## Train

No `get_chat_template()` call here, unlike the Gemma notebook (`get_chat_template(processor,
"gemma-4")`) — the official Qwen3.5 (0.8B) Vision notebook doesn't call it either, so
Qwen3.5's own default chat template is already correct out of the box for this model.

In [ ]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model)  # Enable for training!

# max_length=6144 (not the official example's 2048): the longest page target in this corpus
# runs up to ~4035 chars, since it includes wider 9-col wholesale_retail tables alongside
# 6-col retail_only ones, and Khmer tokenizes less efficiently than Latin text. A v1-era
# Gemma run measured a real char-to-token ratio for this Khmer/Latin-mixed JSON content
# (~1.24 chars/token), which puts the worst case around ~3250 tokens for the target text
# alone, before image tokens + the instruction are added on top -- 6144 leaves real headroom
# above that estimate.
# batch_size=2 (vs Gemma's 1): Unsloth's own default for this much smaller 0.8B model, which
# leaves more headroom on a T4/L4.
trainer = SFTTrainer(
    model = model,
    train_dataset = converted_dataset,
    processing_class = processor.tokenizer,
    data_collator = UnslothVisionDataCollator(model, processor),
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        max_grad_norm = 0.3,
        warmup_ratio = 0.03,
        # -1 / a real int, never None: TrainingArguments._validate_args unconditionally does
        # `max_steps > 0 and num_train_epochs > 0`, so None crashes with a TypeError the
        # moment it's compared. -1 is the library's actual "unset" sentinel for max_steps.
        max_steps = 10 if SMOKE_TEST else -1,
        # See the EPOCHS knob + its reasoning in the data-loading cell above.
        num_train_epochs = 1 if SMOKE_TEST else EPOCHS,
        learning_rate = 2e-4,
        logging_steps = 1,
        save_strategy = "steps",
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
        # You MUST put the below items for vision finetuning (per Unsloth's own notebook):
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = 6144,
    ),
)


In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")


In [ ]:
trainer_stats = trainer.train()

In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
print(f"{trainer_stats.metrics['train_runtime']:.1f}s used for training.")
print(f"Peak reserved memory = {used_memory} GB / {max_memory} GB.")
print(f"Peak reserved memory for training (LoRA) = {used_memory_for_lora} GB.")


In [ ]:
# Push the trained adapter to the Hub RIGHT NOW, before any of the slower eval/diagnostic
# cells below -- a Colab free-tier disconnect (inactivity timeout or session max duration)
# during those cells would otherwise lose this entire training run, since local
# save_pretrained alone doesn't survive past this VM being torn down.
# Requires an HF_TOKEN Colab secret (key icon in the left sidebar) with write access.
from google.colab import userdata

# Versioned by dataset version AND epoch count, so sweep runs (see EPOCHS above) are each
# independently addressable rather than silently overwriting one another or the earlier
# v3-dataset trial (Soxavin/qwen35-ardb-lora-v3, kept as the documented base-model-limitation
# finding, see docs/PROJECT_LOG.md §2.104).
_ADAPTER_REPO_ID = "Soxavin/qwen35-ardb-lora-v5-smoke" if SMOKE_TEST else f"Soxavin/qwen35-ardb-lora-v5-e{EPOCHS}"
_hf_token = userdata.get("HF_TOKEN")
model.push_to_hub(_ADAPTER_REPO_ID, token=_hf_token)
processor.push_to_hub(_ADAPTER_REPO_ID, token=_hf_token)
print(f"Pushed to https://huggingface.co/{_ADAPTER_REPO_ID}")


## Inference sanity check

In [ ]:
from transformers import TextStreamer

FastVisionModel.for_inference(model)  # Enable for inference!

sample = val_dataset[0]
messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": sample["instruction"]}]}]
input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(sample["image"], input_text, add_special_tokens=False, return_tensors="pt").to("cuda")

text_streamer = TextStreamer(processor, skip_prompt=True)
_ = model.generate(**inputs, streamer=text_streamer, max_new_tokens=5000,
                   use_cache=True, temperature=1.0, top_p=0.95, top_k=64)
print("\n--- expected ---\n", sample["text"])


## Evaluate on the validation split (per-label CER + bbox accuracy, not eyeballing)

Each prediction is a JSON list of regions, not one string, so CER against the whole blob
isn't meaningful. Instead: parse both the prediction and the reference, match regions by
`label` (both lists are sorted top-to-bottom, so positional zip within a label stays
correct), then report mean CER per label (skipping `Picture`, whose text is always empty)
plus a bbox accuracy signal (mean per-coordinate absolute difference, 0-1000 scale) across
all matched regions. Malformed JSON is tracked as its own failure count rather than
crashing the loop or silently being dropped.

**Script diagnostic (Khmer vs. Thai), given the known base-model limitation above**: reported
at two levels, not just one whole-row guess — (1) a whole-row check via direct Unicode
codepoint counting (Khmer block vs. Thai block, no extra dependency, exact for this specific
binary disambiguation since the two scripts occupy disjoint code ranges), same as before; and
(2) new — the same codepoint check applied **per region, broken down by label**, so it's
visible whether wrong-script output is uniform across every field or concentrated in specific
ones (e.g. dynamic table content vs. static letterhead text). Cross-checked with
`fast-langdetect` per region as a second, independent signal (a fasttext-based classifier
rather than a fixed Unicode range) — skipped for regions with no alphabetic content (bare
numbers/punctuation), since the library's own accuracy guidance says short/non-linguistic
samples aren't reliably classifiable, and a forced guess there would just add noise.

In [ ]:
import json, re
from fast_langdetect import detect as _langdetect

def levenshtein(a: str, b: str) -> int:
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i] + [0] * len(b)
        for j, cb in enumerate(b, 1):
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb))
        prev = cur
    return prev[-1]

def cer(pred: str, ref: str) -> float:
    return levenshtein(pred, ref) / max(1, len(ref))

def parse_regions(text: str) -> list[dict] | None:
    try:
        regions = json.loads(text)
    except json.JSONDecodeError:
        return None
    return regions if isinstance(regions, list) else None

# Script-detection diagnostic for the known base-model Thai-instead-of-Khmer finding
# (docs/PROJECT_LOG.md §2.104): count Khmer-block vs. Thai-block codepoints directly
# (Unicode ranges, no extra dependency needed just to answer "which script is this") so
# every eval row reports which script it actually generated, not just its CER against a
# Khmer reference (a Thai-script prediction would already show up as ~1.0 CER, but this
# makes the *reason* visible per-row instead of inferring it from a high CER alone).
_KHMER_RE = re.compile(r"[ក-៿]")
_THAI_RE = re.compile(r"[฀-๿]")

def detect_script(text: str) -> str:
    khmer_n, thai_n = len(_KHMER_RE.findall(text)), len(_THAI_RE.findall(text))
    if khmer_n == 0 and thai_n == 0:
        return "neither"
    return "khmer" if khmer_n >= thai_n else "thai"

def detect_lang_fast(text: str) -> str:
    """Secondary cross-check for detect_script(), using fast-langdetect (fasttext-based)
    instead of raw Unicode codepoint counting. Returns "n/a" for text with no alphabetic
    content (bare numbers/punctuation) rather than a misleading guess -- fast-langdetect's
    own docs note accuracy drops on short/non-linguistic samples, and most JSON region
    values here are short."""
    stripped = text.strip()
    if not stripped or not any(ch.isalpha() for ch in stripped):
        return "n/a"
    try:
        result = _langdetect(stripped.replace("\n", " "), model="auto", k=1)
        return result[0]["lang"] if result else "n/a"
    except Exception:
        return "n/a"

def generate(sample):
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": sample["instruction"]}]}]
    input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(sample["image"], input_text, add_special_tokens=False, return_tensors="pt").to("cuda")
    # do_sample=False (greedy): eval numbers must be deterministic run-to-run to compare
    # smoke test vs. full run vs. later epochs. 5000: see the max_length cell above for the
    # char-to-token estimate this margin is based on.
    out = model.generate(**inputs, max_new_tokens=5000, use_cache=True, do_sample=False)
    return processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

cer_by_label: dict[str, list[float]] = {}
bbox_diffs: list[float] = []
parse_failures = 0
row_script_counts: dict[str, int] = {"khmer": 0, "thai": 0, "neither": 0}
region_script_counts: dict[str, dict[str, int]] = {}     # label -> {khmer/thai/neither: n}
region_langdetect_counts: dict[str, dict[str, int]] = {}  # label -> {lang code or "n/a": n}

for i, s in enumerate(val_dataset):
    print(f"[{i + 1}/{len(val_dataset)}] generating doc_id={s['doc_id']} page={s['page']}...")
    pred_text = generate(s)
    row_script_counts[detect_script(pred_text)] += 1
    expected = parse_regions(s["text"]) or []
    predicted = parse_regions(pred_text)
    if predicted is None:
        parse_failures += 1
        continue
    pred_by_label: dict[str, list[dict]] = {}
    for r in predicted:
        pred_by_label.setdefault(r.get("label", ""), []).append(r)
    exp_by_label: dict[str, list[dict]] = {}
    for r in expected:
        exp_by_label.setdefault(r["label"], []).append(r)
    for label, exp_list in exp_by_label.items():
        pred_list = pred_by_label.get(label, [])
        for exp_r, pred_r in zip(exp_list, pred_list):
            pred_region_text = pred_r.get("text", "")
            if label != "Picture" and pred_region_text:
                cer_by_label.setdefault(label, []).append(cer(pred_region_text, exp_r["text"]))
                region_script_counts.setdefault(label, {"khmer": 0, "thai": 0, "neither": 0})
                region_script_counts[label][detect_script(pred_region_text)] += 1
                region_langdetect_counts.setdefault(label, {})
                lang = detect_lang_fast(pred_region_text)
                region_langdetect_counts[label][lang] = region_langdetect_counts[label].get(lang, 0) + 1
            exp_box, pred_box = exp_r.get("box_2d"), pred_r.get("box_2d")
            if exp_box and pred_box and len(exp_box) == 4 and len(pred_box) == 4:
                bbox_diffs.append(sum(abs(a - b) for a, b in zip(exp_box, pred_box)) / 4)

print(f"\nparse failures: {parse_failures} / {len(val_dataset)}")
print(f"script of generated output (whole-row, by codepoint majority): {row_script_counts}")
print("script per region, by label (Unicode codepoint check):")
for label, counts in region_script_counts.items():
    print(f"  {label}: {counts}")
print("language per region, by label (fast-langdetect cross-check; 'n/a' = too short/non-alphabetic to classify):")
for label, counts in region_langdetect_counts.items():
    print(f"  {label}: {counts}")
for label, values in cer_by_label.items():
    print(f"{label}: mean CER = {sum(values) / len(values):.4f} (n={len(values)})")
if bbox_diffs:
    print(f"bbox mean abs diff (0-1000 scale): {sum(bbox_diffs) / len(bbox_diffs):.2f} (n={len(bbox_diffs)})")


## Save

In [ ]:
model.save_pretrained("qwen35_ardb_lora")
processor.save_pretrained("qwen35_ardb_lora")
# model.push_to_hub("your_name/qwen35_ardb_lora", token="YOUR_HF_TOKEN")
# processor.push_to_hub("your_name/qwen35_ardb_lora", token="YOUR_HF_TOKEN")
